In [15]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from datasets import Dataset
from huggingface_hub import hf_hub_download
from transformers import (
    PretrainedConfig, 
    PreTrainedTokenizerFast, 
    Trainer, 
    TrainingArguments,
    DataCollatorWithPadding
)
from transformers.modeling_outputs import SequenceClassifierOutput
from safetensors.torch import load_file
from sklearn.metrics import f1_score, precision_score, recall_score

import zipfile
import json
import torch
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download


In [ ]:
# ==========================================
# 1. PHASE 3 CONFIGURATION & LABELS
# ==========================================
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
MODERN_DECODER_REPO = "A-240242/legal-modern-decoder-checkpoints" 
OUTPUT_DIR = "./phase3_hybrid_encoder_lorma"
HF_TOKENIZER_REPO= "A-240242/legal-modern-decoder-checkpoints"
# Official Phase 3 Labels
LABEL_LIST = [
    "Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", 
    "Dissent", "PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled", 
    "RulingByLowerCourt", "RatioOfTheDecision", "RulingByPresentCourt", "None"
]
NUM_LABELS = len(LABEL_LIST)
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}


In [ ]:
# ==========================================
# 2. DATA LOADING & PREPROCESSING
# ==========================================
def load_and_flatten_rr_data(split_dir, files):
    """Flattens list-based document JSONs into sentence-level (text, label) pairs."""
    sentences = []
    labels = []
    
    for filename in files:
        path = hf_hub_download(
            repo_id=HF_DATASET_REPO, 
            filename=f"Phase_3_RR/{split_dir}/{filename}", 
            repo_type="dataset", 
            token=HF_TOKEN
        )
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
        # Iterate over the list of document dicts directly
        for doc in data:
            doc_texts = doc["text"]
            doc_labels = doc["labels"]
            for text, label in zip(doc_texts, doc_labels):
                sentences.append(text)
                labels.append(LABEL2ID[label])
                
    return sentences, labels

print("Downloading and processing Phase 3 Data...")
train_texts, train_labels = load_and_flatten_rr_data("Train", ["CL_train.json", "IT_train.json"])
dev_texts, dev_labels = load_and_flatten_rr_data("Train", ["CL_dev.json", "IT_dev.json"])
tokenizer = PreTrainedTokenizerFast.from_pretrained(HF_TOKENIZER_REPO, token=os.environ["HF_TOKEN"])
# CRITICAL: For Causal sequence classification, right padding is standard, 
# but we must extract the hidden state of the *actual last token* before padding.
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_data(texts, labels):
    tokenized = tokenizer(texts, padding=False, truncation=True, max_length=512)
    tokenized["labels"] = labels
    return Dataset.from_dict(tokenized)

train_dataset = tokenize_data(train_texts, train_labels)
eval_dataset = tokenize_data(dev_texts, dev_labels)

In [18]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from huggingface_hub import hf_hub_download
from transformers import PretrainedConfig, Trainer, TrainingArguments, DataCollatorWithPadding
from transformers.modeling_outputs import SequenceClassifierOutput
from safetensors.torch import load_file
from sklearn.metrics import f1_score, precision_score, recall_score

# ==========================================
# 1. HELPER CLASSES (RoPE, RMSNorm, SwiGLU)
# ==========================================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm_x = torch.mean(x ** 2, dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(norm_x + self.eps)
        return self.weight * x_normed

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000, device=None):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float().to(device) / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.max_seq_len_cached = max_position_embeddings
        t = torch.arange(self.max_seq_len_cached, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
        self.register_buffer("sin_cached", emb.sin()[None, None, :, :])

    def forward(self, x, seq_len=None):
        if seq_len > self.max_seq_len_cached:
            t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
            freqs = torch.einsum("i,j->ij", t, self.inv_freq)
            emb = torch.cat((freqs, freqs), dim=-1)
            self.register_buffer("cos_cached", emb.cos()[None, None, :, :])
            self.register_buffer("sin_cached", emb.sin()[None, None, :, :])
        return (
            self.cos_cached[:, :, :seq_len, ...].to(dtype=x.dtype),
            self.sin_cached[:, :, :seq_len, ...].to(dtype=x.dtype),
        )

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

class SwiGLUMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class LormaAdapter(nn.Module):
    def __init__(self, hidden_size, rank=128, alpha=128):
        super().__init__()
        self.scaling = alpha / rank
        self.A = nn.Linear(hidden_size, rank, bias=False)
        self.B = nn.Linear(rank, hidden_size, bias=False)
        nn.init.normal_(self.A.weight, std=1e-3)
        nn.init.zeros_(self.B.weight)
    def forward(self, x):
        return self.scaling * self.B(self.A(x))


        
        
# ==========================================
# 1. LORMA (LOW-RANK MATRIX ADAPTATION) WRAPPER
# ==========================================
class LORMALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=16, alpha=32, dropout_p=0.05):
        super().__init__()
        # Frozen Base Layer
        self.base_layer = nn.Linear(in_features, out_features, bias=False)
        self.base_layer.weight.requires_grad = False
        
        # Trainable LORMA Matrices
        self.lorma_A = nn.Linear(in_features, rank, bias=False)
        self.lorma_B = nn.Linear(rank, out_features, bias=False)
        
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(p=dropout_p)
        
        # Initialization
        nn.init.kaiming_uniform_(self.lorma_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B.weight)

    def forward(self, x):
        base_output = self.base_layer(x)
        lorma_output = self.lorma_B(self.lorma_A(self.dropout(x))) * self.scaling
        return base_output + lorma_output

# ==========================================
# 2. HYBRID ENCODER BLOCKS WITH LORMA
# ==========================================
class HybridLORMALocalBlock(nn.Module):
    """Local feature extraction using a BiLSTM (often kept frozen during PEFT)."""
    def __init__(self, config):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_size=config.hidden_size,
            hidden_size=config.hidden_size // 2,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        # x shape: [bsz, seq_len, hidden_size]
        local_features, _ = self.bilstm(x)
        return local_features

class HybridLORMAAttention(nn.Module):
    def __init__(self, config, rank=16, alpha=32):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = self.hidden_size // self.num_heads
        
        # Inject LORMA into all attention projections
        self.q_proj = LORMALinear(self.hidden_size, self.num_heads * self.head_dim, rank=rank, alpha=alpha)
        self.k_proj = LORMALinear(self.hidden_size, self.num_heads * self.head_dim, rank=rank, alpha=alpha)
        self.v_proj = LORMALinear(self.hidden_size, self.num_heads * self.head_dim, rank=rank, alpha=alpha)
        self.o_proj = LORMALinear(self.num_heads * self.head_dim, self.hidden_size, rank=rank, alpha=alpha)

    def forward(self, hidden_states, attention_mask=None):
        bsz, q_len, _ = hidden_states.size()
        
        q = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        attn_mask = None
        if attention_mask is not None:
            attn_mask = attention_mask.unsqueeze(1).unsqueeze(2)
            attn_mask = attn_mask.expand(bsz, self.num_heads, q_len, q_len).bool()

        attn_output = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=False)
        attn_output = attn_output.transpose(1, 2).contiguous().view(bsz, q_len, self.hidden_size)
        
        return self.o_proj(attn_output)

class HybridEncoderLORMALayer(nn.Module):
    def __init__(self, config, rank=16, alpha=32):
        super().__init__()
        self.local_block = HybridLORMALocalBlock(config)
        self.global_attn = HybridLORMAAttention(config, rank=rank, alpha=alpha)
        
        # Inject LORMA into feed-forward block
        self.ffn_up = LORMALinear(config.hidden_size, config.intermediate_size, rank=rank, alpha=alpha)
        self.ffn_down = LORMALinear(config.intermediate_size, config.hidden_size, rank=rank, alpha=alpha)
        
        self.norm1 = nn.LayerNorm(config.hidden_size)
        self.norm2 = nn.LayerNorm(config.hidden_size)

    def forward(self, hidden_states, attention_mask=None):
        # 1. Local Sequential Context
        local_out = self.local_block(hidden_states)
        hidden_states = hidden_states + local_out
        
        # 2. Global Attention Context (LORMA)
        attn_out = self.global_attn(self.norm1(hidden_states), attention_mask)
        hidden_states = hidden_states + attn_out
        
        # 3. FFN (LORMA)
        ffn_out = self.ffn_down(F.gelu(self.ffn_up(self.norm2(hidden_states))))
        hidden_states = hidden_states + ffn_out
        
        return hidden_states

# ==========================================
# 3. SEQUENCE CLASSIFIER & WEIGHT LOADING
# ==========================================
class HybridEncoderLORMAForSequenceClassification(nn.Module):
    def __init__(self, config, num_labels, rank=16, alpha=32):
        super().__init__()
        self.config = config
        self.num_labels = num_labels
        
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=config.pad_token_id)
        self.layers = nn.ModuleList([HybridEncoderLORMALayer(config, rank=rank, alpha=alpha) for _ in range(config.num_hidden_layers)])
        self.norm = nn.LayerNorm(config.hidden_size)
        self.score = nn.Linear(config.hidden_size, num_labels, bias=False)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        hidden_states = self.embed_tokens(input_ids)
        
        for layer in self.layers:
            hidden_states = layer(hidden_states, attention_mask)
            
        hidden_states = self.norm(hidden_states)
        
        # Mean Pooling
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
            sum_embeddings = torch.sum(hidden_states * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            pooled_logits = sum_embeddings / sum_mask
        else:
            pooled_logits = hidden_states.mean(dim=1)

        logits = self.score(pooled_logits)
        
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits.view(-1, self.num_labels), labels.view(-1))
            
        return SequenceClassifierOutput(loss=loss, logits=logits)

In [19]:
# !pip install -q bitsandbytes
# !pip install -q --upgrade torchao

In [24]:
import torch
import torch.nn as nn
import bitsandbytes as bnb
from peft import get_peft_model, LoraConfig, TaskType
from transformers import PretrainedConfig, Trainer, TrainingArguments, DataCollatorWithPadding
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from sklearn.metrics import precision_recall_fscore_support
import math
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HYBRID_ENCODER_REPO = "A-240242/legal-hybridencoder-checkpoints" # Update if your repo differs

print("Fetching Phase 1 Pretrained Hybrid Encoder Checkpoint...")
hybrid_config = PretrainedConfig(
    vocab_size=tokenizer.vocab_size,
    pad_token_id=tokenizer.pad_token_id,
    hidden_size=768,             # UPDATE THIS
    num_hidden_layers=6,         # UPDATE THIS
    num_attention_heads=12,      # UPDATE THIS
    intermediate_size=2048       # UPDATE THIS
)

print("Initializing Hybrid Encoder with LORMA Adapters...")
lorma_hybrid_model = HybridEncoderLORMAForSequenceClassification(hybrid_config, num_labels=NUM_LABELS, rank=16, alpha=32)

# Load base weights (map standard linear layers to the base_layer of LORMALinear)
model_weight_path = hf_hub_download(repo_id=HYBRID_ENCODER_REPO, filename="model.safetensors", token=HF_TOKEN)
state_dict = load_file(model_weight_path)

mapped_state_dict = {}
for key, tensor in state_dict.items():
    if "proj.weight" in key or "ffn_up.weight" in key or "ffn_down.weight" in key:
        mapped_state_dict[key.replace(".weight", ".base_layer.weight")] = tensor
    else:
        mapped_state_dict[key] = tensor

lorma_hybrid_model.load_state_dict(mapped_state_dict, strict=False)

# Freeze entire model, then unfreeze only LORMA matrices and classifier head
for param in lorma_hybrid_model.parameters(): 
    param.requires_grad = False

for name, param in lorma_hybrid_model.named_parameters():
    if "lorma_A" in name or "lorma_B" in name or "score" in name:
        param.requires_grad = True

lorma_hybrid_model.to(device)
trainable_params = sum(p.numel() for p in lorma_hybrid_model.parameters() if p.requires_grad)
print(f"Trainable LORMA Parameters: {trainable_params:,}")



Fetching Phase 1 Pretrained Hybrid Encoder Checkpoint...
Initializing Hybrid Encoder with LORMA Adapters...


model.safetensors: reconstructing file:   0%|          |  0.00B /  335MB            

model.safetensors: downloading bytes:           |  0.00B            

Trainable LORMA Parameters: 1,140,480


In [25]:
# 4. Restore Collator and Metrics
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
        
    predictions = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    return {
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }

# 5. Execute QLoRA Training
training_args_lorma = TrainingArguments(
    output_dir="./phase3_hybrid_encoder_lorma",
    report_to="none",
    num_train_epochs=10, 
    per_device_train_batch_size=16, 
    gradient_accumulation_steps=2, 
    learning_rate=3e-4, 
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="macro_f1",
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    seed=42
)

lorma_trainer = Trainer(
    model=lorma_hybrid_model,
    args=training_args_lorma,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator, # Ensure data_collator is defined from your previous cell
    compute_metrics=compute_metrics # Ensure compute_metrics is defined
)

print("\nCommencing Hybrid Encoder LORMA Training...")
lorma_trainer.train()


Commencing Hybrid Encoder LORMA Training...


Epoch,Training Loss,Validation Loss,Macro Precision,Macro Recall,Macro F1
1,3.252904,1.856767,0.270765,0.214107,0.206781
2,2.675092,1.813582,0.376919,0.280649,0.297171
3,2.175405,1.973539,0.370705,0.279835,0.291028
4,1.864022,2.146746,0.382790,0.256260,0.279215
5,1.320799,2.561763,0.363128,0.252490,0.259883
6,0.998369,2.991488,0.340700,0.255213,0.270029
7,0.683461,3.389219,0.291460,0.256948,0.262796
8,0.446644,3.963292,0.291807,0.251051,0.257760
9,0.285582,4.375545,0.298775,0.264704,0.270777
10,0.180642,4.577055,0.300880,0.266605,0.273469


TrainOutput(global_step=5270, training_loss=1.4126346784931885, metrics={'train_runtime': 1025.8514, 'train_samples_per_second': 164.205, 'train_steps_per_second': 5.137, 'total_flos': 0.0, 'train_loss': 1.4126346784931885, 'epoch': 10.0})

In [26]:
# ==========================================
# 6. VALIDATION SET EVALUATION
# ==========================================
print("Running evaluation on the Development Set...")
eval_results = lorma_trainer.evaluate()
print(f"Validation Macro-F1: {eval_results['eval_macro_f1']:.4f}")
print(f"Validation Precision: {eval_results['eval_macro_precision']:.4f}")
print(f"Validation Recall: {eval_results['eval_macro_recall']:.4f}")

Running evaluation on the Development Set...


Training Loss,Validation Loss,Epoch,Macro Precision,Macro Recall,Macro F1
0.180642,1.813582,10,0.376919,0.280649,0.297171


Validation Macro-F1: 0.2972
Validation Precision: 0.3769
Validation Recall: 0.2806


In [27]:
# ==========================================
# 7. LOAD TEST DATA
# ==========================================
def load_test_data(files):
    test_docs = []
    for filename in files:
        path = hf_hub_download(
            repo_id=HF_DATASET_REPO, 
            filename=f"Phase_3_RR/Test/{filename}", 
            repo_type="dataset", 
            token=HF_TOKEN
        )
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
            if isinstance(data, dict):
                for doc_id, content in data.items():
                    # If the content is just a list of sentences
                    if isinstance(content, list):
                        test_docs.append({"id": doc_id, "text": content})
                    # If it happens to be a dictionary
                    elif isinstance(content, dict):
                        if "id" not in content:
                            content["id"] = doc_id
                        test_docs.append(content)
                        
            elif isinstance(data, list):
                test_docs.extend(data)
                
    return test_docs

print("\nDownloading Test Data...")
test_docs = load_test_data(["CL_test.json", "IT_test.json"])
print(f"Successfully loaded {len(test_docs)} test documents.")


Successfully loaded 10 test documents.


In [28]:
# ==========================================
# 8. TEST SET INFERENCE & SUBMISSION
# ==========================================
lorma_hybrid_model .eval()
lorma_hybrid_model .to(device)
predictions = {}

print("Generating Rhetorical Role Predictions...")
for doc in tqdm(test_docs, desc="Predicting Document Sentences"):
    doc_id = doc["id"]
    sentences = doc["text"]
    doc_predictions = []
    
    batch_size = 16
    for i in range(0, len(sentences), batch_size):
        batch_sentences = sentences[i : i + batch_size]
        
        inputs = tokenizer(
            batch_sentences, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors="pt"
        ).to(device)
        
        with torch.inference_mode():
            outputs = lorma_hybrid_model(**inputs)
            preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            
        doc_predictions.extend([ID2LABEL[pred] for pred in preds])
        
    assert len(doc_predictions) == len(sentences), f"Count mismatch for {doc_id}"
    predictions[doc_id] = doc_predictions

Generating Rhetorical Role Predictions...


Predicting Document Sentences:   0%|          | 0/10 [00:00<?, ?it/s]

In [29]:
# ==========================================
# 9. SAVE & ZIP FOR CODABENCH
# ==========================================
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print(f"\nPhase 3 Predictions complete! Submission saved to: {SUBMISSION_ZIP}")


Phase 3 Predictions complete! Submission saved to: ./phase3_hybrid_encoder_lorma/submission.zip
